# HMP-GNN — Google Colab

联邦学习 + HMP-GAE 防御 + V2 M7 指标（CSE / PPL）。**本 notebook 不再内联实验超参**；**唯一权威配置在仓库根目录的 `main.py` 的 `main()` 里**（`experiment_name`、轮数、模型、`defense_config` 等）。

## 使用说明

1. **GPU**：**Runtime → Change runtime type** → 选 T4 等 GPU。
2. 若只上传了本 notebook，请先跑 **Step 0** 克隆代码并 `cd` 到仓库根。
3. 可选：在 **Step 3** 里填写 **`COLAB_CONFIG_OVERRIDES`**，与 `bakcup_for_reference/GRMP_Attack_Colab.ipynb` 相同规范；**`{}` 表示完全使用 `main.py` 默认**。
4. **主实验**由 `from main import main; main(...)` 驱动，与 `main.py` 中 `config` 一致。结束后在 `results/` 下展示 `experiment_name_figure1.png` … `figure5.png`（前缀与 `main.py` 中 `experiment_name` 一致）。
5. **（可选）** 若需要三场景 + Fig A/C/E/F，可运行 `run_demo.py`；该脚本为快速演示有**独立**轻量配置，**论文主结果以 `main.py` 为准**。
6. 最后运行 **「释放 GPU 并断开运行时」** 单元，避免空挂算力。

## Step 0: 获取代码

若当前目录下已有 `main.py`，不克隆；否则拉取并进入 `HMP-GNN`。

In [ ]:
# Fetch repository and set working directory
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/GuangLun2000/HMP-GNN.git"
REPO_DIR = Path("HMP-GNN")

def code_files_present():
    return Path("main.py").is_file() and Path("data_loader.py").is_file()

if code_files_present():
    print("已在仓库根目录（发现 main.py）。")
else:
    if REPO_DIR.is_dir():
        print(f"使用已有目录: {REPO_DIR}")
    else:
        print(f"正在克隆 {REPO_URL} ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir(REPO_DIR)
    print(f"工作目录: {Path('.').resolve()}")

sys.path.insert(0, str(Path('.').resolve()))
print("cwd:", Path('.').resolve())


## Step 1: 安装依赖

In [ ]:
from pathlib import Path
req = Path("requirements.txt")
if req.is_file():
    print("Installing from requirements.txt ...")
    %pip install -q -r requirements.txt
else:
    %pip install -q torch transformers datasets numpy scikit-learn pandas tqdm matplotlib seaborn peft
print("依赖已安装")


## Step 2: 检查文件与 GPU

In [ ]:
import os
from pathlib import Path
import torch

required = [
    "main.py", "client.py", "server.py", "data_loader.py", "models.py", "visualization.py",
    "defense.py", "fed_checkpoint.py", "run_demo.py", "attack_baseline_hallucination.py", "hmp_gae/__init__.py",
]
missing = [f for f in required if not os.path.exists(f)]
if missing:
    print("缺少文件:", missing)
else:
    print("核心文件已就绪")
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("未检测到 GPU。Runtime -> Change runtime type -> GPU。")


## Step 3: 运行主实验

**参数**仅来自 `main.py` 的 `config`，此处仅有可选的 **`COLAB_CONFIG_OVERRIDES`**。

In [ ]:
# 可选：合并进 main() 的 config；空 {} = 使用 main.py 原样
COLAB_CONFIG_OVERRIDES = {
    # "experiment_name": "my_colab_run",
    # "num_rounds": 5,
    # "dataset_size_limit": 8000,
}

print("Configuration: main.py 内 config")
if COLAB_CONFIG_OVERRIDES:
    print("  + Colab 覆写键:", list(COLAB_CONFIG_OVERRIDES.keys()))
else:
    print("  + Colab 覆写: 无")


In [ ]:
import os
import gc
import warnings
import torch
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from main import main

print("启动实验（main.py 默认 config，除非 COLAB_CONFIG_OVERRIDES 非空）")
print("=" * 60)
try:
    main(config_overrides=COLAB_CONFIG_OVERRIDES if COLAB_CONFIG_OVERRIDES else None)
    print()
    print("实验完成")
except Exception as e:
    print()
    print("失败:", e)
    import traceback
    traceback.print_exc()
finally:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("gc + CUDA empty_cache 已执行")


## Step 4: 主实验图 (Figure 1–5)

文件名形如 `{experiment_name}_figure1.png`，与 `main.py` 中 `experiment_name` 一致。

In [ ]:
from IPython.display import Image, display, Markdown
from pathlib import Path
import json

results_dir = Path("results")
_all = list(results_dir.glob("*_results.json"))
candidates = [p for p in _all if not p.name.startswith("v1demo_")]
if not candidates:
    candidates = _all
candidates = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)
if not candidates:
    display(Markdown("未找到 results/*_results.json（主实验是否已跑？）"))
else:
    with open(candidates[0], "r", encoding="utf-8") as f:
        bundle = json.load(f)
    exp = (bundle.get("config") or {}).get("experiment_name", "experiment")
    display(Markdown("**结果中的 experiment_name:** `" + str(exp) + "`"))
    for i in range(1, 6):
        p = results_dir / f"{exp}_figure{i}.png"
        if p.is_file():
            display(Markdown("#### Figure " + str(i) + " -- `" + p.name + "`"))
            display(Image(filename=str(p)))
        else:
            display(Markdown("#### Figure " + str(i) + " -- 未找到 `" + str(p) + "`"))


## （可选）三场景 + Fig A / C / E / F

`run_demo.py` 有独立快配置；主论文以 `main.py` 为准。

In [ ]:
!python run_demo.py


In [ ]:
from IPython.display import Image, display, Markdown
from pathlib import Path

FIG_DIR = Path("results/_v1_demo")
KEY = [
    ("Fig A", "figA_defense_bar.png"),
    ("Fig C", "figC_trust_evolution.png"),
    ("Fig E", "figE_metrics_grouped.png"),
    ("Fig F", "figF_cse_evolution.png"),
]
if not FIG_DIR.is_dir():
    display(Markdown("未找到 results/_v1_demo（可跳过可选单元或先运行 run_demo）。"))
else:
    for title, name in KEY:
        p = FIG_DIR / name
        if p.is_file():
            display(Markdown("### " + title + " -- `" + str(p) + "`"))
            display(Image(filename=str(p)))
        else:
            display(Markdown("### " + title + " -- 缺 " + name))


## 数字摘要

In [ ]:
import json
from pathlib import Path
from typing import Optional
from visualization import summarize_run_multi_metric

def show_one(res_path: Path, ppl_path: Optional[Path], label: str) -> None:
    if not res_path.is_file():
        print(label + ": 缺 " + str(res_path))
        return
    s = summarize_run_multi_metric(
        res_path, ppl_json_path=ppl_path if ppl_path and ppl_path.is_file() else None
    )
    acc = s.get("final_clean_acc", 0.0)
    cse = s.get("mean_cse")
    ppl = s.get("ppl")
    cse_s = "%.4f" % cse if cse is not None else "N/A"
    ppl_s = "%.2f" % ppl if ppl is not None else "N/A"
    print("%-32s  acc=%.4f  mean_cse=%s  ppl=%s" % (label, acc, cse_s, ppl_s))

R = Path("results")
_allj = list(R.glob("*_results.json"))
rjs = [p for p in _allj if not p.name.startswith("v1demo_")]
if not rjs:
    rjs = _allj
rjs = sorted(rjs, key=lambda p: p.stat().st_mtime, reverse=True)
if rjs:
    rj = rjs[0]
    with open(rj, "r", encoding="utf-8") as f:
        cfg = json.load(f).get("config", {})
    exp = cfg.get("experiment_name", rj.stem.replace("_results", ""))
    ppl = R / f"{exp}_eval_ppl.json"
    print("-- 主实验 (main) --")
    show_one(rj, ppl, exp)
print("-- run_demo 三场景（若存在）--")
for label, stem in [
    ("NoAttack", "v1demo_noattack"),
    ("Hallu+FedAvg", "v1demo_hallu_fedavg"),
    ("Hallu+HMP-GAE", "v1demo_hallu_hmpgae"),
]:
    show_one(R / f"{stem}_results.json", R / f"{stem}_eval_ppl.json", label)


## 打包下载 results 产物

In [ ]:
import shutil
from pathlib import Path

out_dir = Path("results/hmp_gae_colab_bundle")
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)
root = Path("results")
for p in list(root.rglob("*")):
    if p.is_file() and p.suffix in {".png", ".pdf", ".json", ".jsonl", ".csv"}:
        if "hmp_gae_colab_bundle" in p.parts:
            continue
        dest = out_dir / p.relative_to(root)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dest)
zip_path = str(shutil.make_archive(str(out_dir), "zip", root_dir=out_dir))
print("Zip:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("非 Colab 或未自动下载, zip 路径:", zip_path, type(e).__name__)


## 释放 GPU 并断开 Colab 运行时

**务必运行下一单元**，避免长时间占用 GPU。

In [ ]:
import gc, time
import torch

print("准备释放显存并断开本 Colab 运行时…")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    try:
        torch.cuda.synchronize()
    except Exception:
        pass
time.sleep(2)
try:
    from google.colab import runtime
    print("约 20 秒后 runtime.unassign() 释放实例。")
    time.sleep(20)
    runtime.unassign()
except Exception as e:
    print("非 Colab 或 unassign 不可用:", type(e).__name__, e)
    print("可手动: Runtime -> Disconnect and delete runtime")
